In [1]:
!pip install -q pymupdf pandas scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 26.2 MB/s eta 0:00:00


In [2]:
import os
import re
import fitz
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

print("Libraries loaded successfully.")

Libraries loaded successfully.


In [4]:
#Mount Google Drive and define paths
from google.colab import drive

drive.mount('/content/drive')

PROJECT_PATH = "/content/drive/MyDrive/research paper project"
PDF_FOLDER = os.path.join(PROJECT_PATH, "RP pdf")
CSV_PATH = os.path.join(PROJECT_PATH, "research_paper_insights.csv")

print("Project folder:", PROJECT_PATH)
print("PDF folder:", PDF_FOLDER)
print("CSV output:", CSV_PATH)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Project folder: /content/drive/MyDrive/research paper project
PDF folder: /content/drive/MyDrive/research paper project/RP pdf
CSV output: /content/drive/MyDrive/research paper project/research_paper_insights.csv


In [5]:
#Check that the PDFs exist

pdf_files = {
    "RP1": os.path.join(PDF_FOLDER, "RP1.pdf"),
    "RP2": os.path.join(PDF_FOLDER, "RP2.pdf"),
    "RP3": os.path.join(PDF_FOLDER, "RP3.pdf"),
    "RP4": os.path.join(PDF_FOLDER, "RP4.pdf"),
    "RP5": os.path.join(PDF_FOLDER, "RP5.pdf")
}

for paper_id, path in pdf_files.items():
    if os.path.exists(path):
        print(f"{paper_id}: Found")
    else:
        print(f"{paper_id}: NOT FOUND")

RP1: Found
RP2: Found
RP3: Found
RP4: Found
RP5: Found


In [24]:
#Extract text from PDFs

def extract_pdf_text(pdf_path):

    doc = fitz.open(pdf_path)

    pages_text = []

    for page in doc:

        text = page.get_text()

        # Preserve hyphens when words are split across lines
        text = re.sub(r'-\s*\n\s*', '-', text)

        # Replace remaining line breaks with spaces
        text = re.sub(r'\s*\n\s*', ' ', text)

        # Remove excessive spaces
        text = re.sub(r'\s+', ' ', text).strip()

        pages_text.append(text)

    full_text = " ".join(pages_text)

    return full_text, len(doc)

In [7]:
#Extract abstracts
def extract_abstract(text):

    # Look for ABSTRACT heading
    match = re.search(
        r'\bABSTRACT\b\s*:?\s*(.*?)(?=\n\s*(?:1\.?\s+INTRODUCTION|INTRODUCTION|Keywords?|Index Terms?|CCS CONCEPTS)\b)',
        text,
        re.IGNORECASE | re.DOTALL
    )

    if match:
        abstract = match.group(1).strip()
        abstract = re.sub(r'\s+', ' ', abstract)

        if len(abstract) > 100:
            return abstract

    # Fallback: take text after ABSTRACT
    match = re.search(
        r'\bABSTRACT\b\s*:?\s*(.*)',
        text,
        re.IGNORECASE | re.DOTALL
    )

    if match:
        abstract = match.group(1).strip()

        # Stop at likely next section
        abstract = re.split(
            r'\n\s*(?:Keywords?|Index Terms?|1\.?\s+INTRODUCTION|INTRODUCTION)\b',
            abstract,
            flags=re.IGNORECASE
        )[0]

        abstract = re.sub(r'\s+', ' ', abstract).strip()

        if len(abstract) > 100:
            return abstract

    return "Not available"

In [9]:
#Extract keywords
def extract_keywords(text):

    # Look for Index Terms / Keywords
    match = re.search(
        r'(?:Index Terms?|Keywords?)\s*[:\-]?\s*(.*?)(?=\n\s*(?:1\.?\s+INTRODUCTION|INTRODUCTION|I\.?\s+INTRODUCTION)\b)',
        text,
        re.IGNORECASE | re.DOTALL
    )

    if match:
        keywords = match.group(1).strip()

        # Clean line breaks
        keywords = re.sub(r'\s+', ' ', keywords)

        # Remove trailing punctuation
        keywords = keywords.strip(" .;:")

        if keywords:
            return keywords

    return "Not provided"

In [10]:
#Generate extractive summary
def generate_summary(text, num_sentences=3):

    # Clean text
    clean_text = re.sub(r'\s+', ' ', text).strip()

    # Split into sentences
    sentences = re.split(r'(?<=[.!?])\s+', clean_text)

    # Remove extremely short sentences
    sentences = [
        sentence.strip()
        for sentence in sentences
        if len(sentence.strip().split()) >= 8
    ]

    if len(sentences) <= num_sentences:
        return " ".join(sentences)

    try:
        vectorizer = TfidfVectorizer(
            stop_words='english',
            max_features=5000
        )

        matrix = vectorizer.fit_transform(sentences)

        # Document representation
        document_vector = matrix.mean(axis=0)

        # Similarity of each sentence to overall document
        similarities = cosine_similarity(
            matrix,
            document_vector
        ).flatten()

        # Select highest scoring sentences
        top_indices = similarities.argsort()[-num_sentences:]

        # Restore original order
        top_indices = sorted(top_indices)

        summary = " ".join(
            sentences[i]
            for i in top_indices
        )

        return summary

    except Exception:
        return " ".join(sentences[:num_sentences])


In [11]:
#Generate TF-IDF keywords
def extract_tfidf_keywords(text, top_n=8):

    clean_text = re.sub(r'\s+', ' ', text).strip()

    try:
        vectorizer = TfidfVectorizer(
            stop_words='english',
            max_features=5000
        )

        matrix = vectorizer.fit_transform([clean_text])

        scores = matrix.toarray()[0]
        terms = vectorizer.get_feature_names_out()

        ranked_indices = scores.argsort()[::-1]

        keywords = [
            terms[i]
            for i in ranked_indices[:top_n]
            if scores[i] > 0
        ]

        return ", ".join(keywords)

    except Exception:
        return "Not available"


In [12]:
#Validated titles and authors

clean_titles = {
    "RP1.pdf": "Epidemiological Causal Graph Identification: Challenges, Identifiability and Algorithms",

    "RP2.pdf": "Score Centering Stabilizes Off-Policy Reinforcement Learning",

    "RP3.pdf": "Paint-Anything: Unified Any-Color Control for Image Generation and Editing",

    "RP4.pdf": "Towards AI-enhanced control: a numerical technique for trajectory smoothing of a parallel robot for pancreatic surgery",

    "RP5.pdf": "Detecting Deceptive Recruitment: A Signal-theoretic Machine Learning Framework for Early Identification of Labour Exploitation"
}


clean_authors = {
    "RP1.pdf": "Sambit Mishra, Yingying Wang, Christine K. Johnson, Urbashi Mitra",

    "RP2.pdf": "Martin Marek, Max Ryabinin",

    "RP3.pdf": "Ji Xie, Dewei Zhou, Xinyu Huang, Zhennan Chen, Xun Wang",

    "RP4.pdf": "Iosif Birlescu, Alexandru Pusca, Bogdan Gherman, Calin Vaida, Ionut Zima, Damien Chablat, Doina Pisla",

    "RP5.pdf": "Sajid Siraj, Mahnaz Hosseinzadeh, Amin Vafadarnikjoo, Shuyang Li"
}

print("Validated metadata loaded.")

Validated metadata loaded.


In [26]:
#claude given code
import re
try:
    import pymupdf as fitz
except ImportError:
    import fitz

AFFIL_WORDS = {"university","universität","université","institute","institut","department","dept","school",
    "laboratory","lab","college","academy","center","centre","faculty","cnrs","corp","inc","ltd","gmbh",
    "research","seed","labs","foundation","hospital","technology","technical","sciences","ecole","école","group"}
PARTICLES = {"de","van","von","der","den","la","le","da","di","del","bin","al","el","ben"}
SMALL = {"a","an","the","and","or","of","for","via","in","on","to","with","by","at","vs","from","as"}
ACRONYMS = {"ai","rl","llm","llms","nlp","ml","gpu","t2i","mis","sar"}
NAME_TOKEN = re.compile(r"^(?:[A-ZÀ-ÖØ-Þ][A-Za-zà-öø-ÿ'’\-]*[a-zà-öø-ÿ]|[A-ZÀ-ÖØ-Þ]\.?)$")

def _line_text(l, max_size):
    """Rebuild text from characters, dropping superscripts and re-inserting missing spaces."""
    out, prev_x1 = [], None
    for s in l["spans"]:
        is_sup = bool(s["flags"] & 1) or (s["size"] < 0.72 * max_size and len("".join(c["c"] for c in s["chars"]).strip()) <= 6)
        if is_sup:
            prev_x1 = None if prev_x1 is None else prev_x1 + 999   # force a gap after a dropped superscript
            continue
        for ch in s["chars"]:
            c = ch["c"]
            if c == " ":
                out.append(" "); prev_x1 = ch["bbox"][2]; continue
            if prev_x1 is not None and ch["bbox"][0] - prev_x1 > 0.15 * s["size"] and out and out[-1] != " ":
                out.append(" ")
            out.append(c); prev_x1 = ch["bbox"][2]
    return re.sub(r"\s+", " ", "".join(out)).strip()

def get_first_page_lines(pdf_path):
    page = fitz.open(pdf_path)[0]
    H = page.rect.height
    raw = []
    for b in page.get_text("rawdict")["blocks"]:
        if b.get("type") != 0:
            continue
        for l in b["lines"]:
            if abs(l["dir"][0]) < 0.99:                    # skip rotated text (arXiv side stamp)
                continue
            spans = [s for s in l["spans"] if "".join(c["c"] for c in s["chars"]).strip()]
            if not spans or l["bbox"][1] > 0.6 * H:
                continue
            mx = max(s["size"] for s in spans)
            text = _line_text(l, mx)
            if text:
                raw.append({"text": text, "size": mx, "y0": l["bbox"][1], "y1": l["bbox"][3], "x0": l["bbox"][0]})
    # merge fragments on the same visual row (small-caps titles are split into pieces)
    raw.sort(key=lambda d: d["y0"])
    rows = []
    for ln in raw:
        if rows:
            r = rows[-1]
            overlap = min(r["y1"], ln["y1"]) - max(r["y0"], ln["y0"])
            if overlap >= 0.5 * min(r["y1"] - r["y0"], ln["y1"] - ln["y0"]):
                r["parts"].append(ln)
                r["y0"], r["y1"] = min(r["y0"], ln["y0"]), max(r["y1"], ln["y1"])
                continue
        rows.append({"y0": ln["y0"], "y1": ln["y1"], "parts": [ln]})
    lines = []
    for r in rows:
        parts = sorted(r["parts"], key=lambda d: d["x0"])
        lines.append({"text": " ".join(p["text"] for p in parts), "size": max(p["size"] for p in parts),
                      "y0": r["y0"], "y1": r["y1"], "x0": parts[0]["x0"]})
    return lines

def find_title_lines(lines):
    groups, cur = [], []
    for ln in lines:
        if cur and ln["size"] >= 0.75 * max(c["size"] for c in cur) and ln["y0"] - cur[-1]["y1"] < 0.6 * ln["size"]:
            cur.append(ln)
        else:
            if cur: groups.append(cur)
            cur = [ln]
    if cur: groups.append(cur)
    ok = [g for g in groups
          if sum(len(l["text"].split()) for l in g) >= 3
          and not re.search(r"arxiv|preprint|under review", " ".join(l["text"] for l in g), re.I)]
    return max(ok, key=lambda g: max(l["size"] for l in g), default=[])

def smart_title(s):
    out = []
    for i, w in enumerate(s.lower().split(" ")):
        parts = []
        for p in w.split("-"):
            if p in ACRONYMS: parts.append(p.upper())
            elif i > 0 and p in SMALL: parts.append(p)
            else: parts.append(p[:1].upper() + p[1:])
        out.append("-".join(parts))
    return " ".join(out)

def join_title(tlines):
    out = ""
    for l in tlines:
        t = l["text"]
        out = out + t if out.endswith("-") else (out + " " + t if out else t)
    out = re.sub(r"\s+", " ", out).strip()
    return smart_title(out) if out.isupper() else out

def split_names(text):
    s = re.sub(r"\[\s*\d{4}-\d{4}-\d{4}-\d{3}[\dXx]\s*\]", "", text)       # ORCID ids
    s = re.sub(r"\S+@\S+", "", s)                                            # emails
    s = re.sub(r"[∗*†‡§¶#⋆¹²³⁴⁵⁶⁷⁸⁹⁰]", "", s)                                # markers
    s = re.sub(r"(?<=[^\W\d_])\d+(?:\s*,\s*\d+)*", "", s)                    # affiliation numbers
    s = re.sub(r"\s+(?:and|&)\s+", ", ", s)
    return [p.strip(" ;") for p in s.split(",") if p.strip(" ;")]

def is_name(p):
    toks = p.split()
    if not 2 <= len(toks) <= 5: return False
    if any(t.lower().strip(".,") in AFFIL_WORDS for t in toks): return False
    return all(NAME_TOKEN.match(t) or t.lower() in PARTICLES for t in toks)

def extract_authors(lines, tlines):
    after = [l for l in lines if l["y0"] >= tlines[-1]["y1"] - 1]
    block, prev = "", ""
    for l in after[:10]:
        t = l["text"]
        if re.match(r"(?i)^(abstract|keywords|index terms|1\.?\s*introduction|introduction)", t):
            break
        continues = prev.endswith("-")                        # name split across lines: "Gher-" / "man"
        has_name = any(is_name(p) for p in split_names(t))
        if not (has_name or continues):
            if block: break
            continue
        block = block[:-1] + t if continues else (block + " " + t).strip()
        prev = t
    names = [p for p in split_names(block) if is_name(p)]
    return ", ".join(dict.fromkeys(names)) or "Not available"

def extract_title_authors(pdf_path):
    lines = get_first_page_lines(pdf_path)
    tl = find_title_lines(lines)
    if tl:
        return join_title(tl), extract_authors(lines, tl)
    meta = fitz.open(pdf_path).metadata or {}
    return (meta.get("title") or "Not available"), (meta.get("author") or "Not available")

In [13]:
#Extract preprint dates
def extract_preprint_date(text):

    pattern = (
        r"arXiv:\S+.*?"
        r"(\d{1,2}\s+[A-Za-z]{3}\s+\d{4})"
    )

    match = re.search(
        pattern,
        text,
        re.IGNORECASE | re.DOTALL
    )

    if match:
        return match.group(1)

    return "Not available"

In [37]:
#Process all 5 papers
records = []

for paper_id, pdf_path in pdf_files.items():

    file_name = os.path.basename(pdf_path)

    print(f"Processing {paper_id}...")

    # Extract PDF text
    text, page_count = extract_pdf_text(pdf_path)

    # Extract abstract
    abstract = extract_abstract(text)

    # Extract keywords
    keywords = extract_keywords(text)

    # Generate summary
    summary = generate_summary(abstract)

    # Generate TF-IDF keywords
    tfidf_keywords = extract_tfidf_keywords(abstract)

    # Word count
    word_count = len(text.split())

    # Preprint date
    preprint_date = extract_preprint_date(text)

    # Automatically extract title and authors
title, authors = extract_title_authors(pdf_path)

# Fallback to validated metadata if automatic extraction fails
if title == "Not available":
    title = clean_titles.get(file_name, "Not available")

if authors == "Not available":
    authors = clean_authors.get(file_name, "Not available")

record = {
    "Paper_ID": paper_id,
    "File_Name": file_name,
    "Title": title,
    "Preprint_Date": preprint_date,
    "Publication_Year": None,
    "Abstract": abstract,
    "Keywords": keywords,
    "Pages": page_count,
    "Word_Count": word_count,
    "Authors": authors,
    "Summary": summary,
    "TFIDF_Keywords": tfidf_keywords
}

records.append(record)

print(f"✓ {paper_id} completed")

Processing RP1...
Processing RP2...
Processing RP3...
Processing RP4...
Processing RP5...
✓ RP5 completed


In [15]:
#Display the dataset
pd.set_option('display.max_colwidth', 100)

papers_df

,Paper_ID,File_Name,Title,Preprint_Date,Publication_Year,Abstract,Keywords,Pages,Word_Count,Authors,Summary,TFIDF_Keywords
0,RP1,RP1.pdf,"Epidemiological Causal Graph Identification: Challenges, Identifiability and Algorithms",17 Sep 2026,None,"—Causal discovery from observational data is fundamental to statistics and machine learning, yet...","—Causal Discovery, Directed Acyclic Graphs, Identifiability, Exponential Family, Ordinal Distrib...",5,3938,"Sambit Mishra, Yingying Wang, Christine K. Johnson, Urbashi Mitra","Epidemiological Causal Graph Identification: Challenges, Identifiability and Algorithms Sambit M...","ordinal, family, causal, exponential, continuous, discovery, models, graphs"
1,RP2,RP2.pdf,Score Centering Stabilizes Off-Policy Reinforcement Learning,17 Sep 2026,None,Reinforcement learning (RL) of large language models is notoriously sensitive to small differenc...,Not provided,16,8712,"Martin Marek, Max Ryabinin",SCORE CENTERING STABILIZES OFF-POLICY REINFORCEMENT LEARNING Martin Marek & Max Ryabinin Togethe...,"training, sampling, score, quantization, importance, correction, centering, sc"
2,RP3,RP3.pdf,Paint-Anything: Unified Any-Color Control for Image Generation and Editing,17 Sep 2026,None,Professional design requires any-color control: the ability to specify an object’s target color ...,Not provided,29,13061,"Ji Xie, Dewei Zhou, Xinyu Huang, Zhennan Chen, Xun Wang","Paint-Anything: Unified Any-Color Control for Image Generation and Editing Ji Xie1,2, Dewei Zhou...","color, hex, acbench, editing, object, paint, generation, models"
3,RP4,RP4.pdf,Towards AI-enhanced control: a numerical technique for trajectory smoothing of a parallel robot ...,Not available,None,. The paper presents a numerical approach for the end-effector trajectory smoothing of a paralle...,"Trajectory Smoothing, Parallel Robot, Minimally Invasive Surgery",11,4466,"Iosif Birlescu, Alexandru Pusca, Bogdan Gherman, Calin Vaida, Ionut Zima, Damien Chablat, Doina ...",Towards AI-enhanced control: a numerical technique for trajectory smoothing of a parallel robot ...,"approach, end, command, space, effector, time, control, based"
4,RP5,RP5.pdf,Detecting Deceptive Recruitment: A Signal-theoretic Machine Learning Framework for Early Identif...,17 Sep 2026,None,"Deceptive online job advertisements have emerged as a primary pathway into forced labour, yet sy...","density, and visa sponsorship mentions ranking highest, followed by visual colour and texture fe...",32,10899,"Sajid Siraj, Mahnaz Hosseinzadeh, Amin Vafadarnikjoo, Shuyang Li",Detecting Deceptive Recruitment: A Signal-theoretic Machine Learning Framework for Early Identif...,"detection, risk, primary, exploiters, data, language, quality, legitimate"


In [16]:
#Quality check
print("===== DATA QUALITY CHECK =====")

print("\nNumber of rows:", len(papers_df))
print("Number of columns:", len(papers_df.columns))

print("\nMissing values:")
print(papers_df.isnull().sum())

print("\nDuplicate Paper IDs:")
print(papers_df["Paper_ID"].duplicated().sum())

print("\nEmpty abstracts:")
print(
    (papers_df["Abstract"].fillna("").str.strip() == "").sum()
)

print("\nEmpty summaries:")
print(
    (papers_df["Summary"].fillna("").str.strip() == "").sum()
)

print("\nWord counts:")
print(papers_df[["Paper_ID", "Word_Count"]])

print("\nTF-IDF keywords:")
print(papers_df[["Paper_ID", "TFIDF_Keywords"]])

===== DATA QUALITY CHECK =====

Number of rows: 5
Number of columns: 12

Missing values:
Paper_ID            0
File_Name           0
Title               0
Preprint_Date       0
Publication_Year    5
Abstract            0
Keywords            0
Pages               0
Word_Count          0
Authors             0
Summary             0
TFIDF_Keywords      0
dtype: int64

Duplicate Paper IDs:
0

Empty abstracts:
0

Empty summaries:
0

Word counts:
  Paper_ID  Word_Count
0      RP1        3938
1      RP2        8712
2      RP3       13061
3      RP4        4466
4      RP5       10899

TF-IDF keywords:
  Paper_ID  \
0      RP1   
1      RP2   
2      RP3   
3      RP4   
4      RP5   

                                                                   TFIDF_Keywords  
0     ordinal, family, causal, exponential, continuous, discovery, models, graphs  
1  training, sampling, score, quantization, importance, correction, centering, sc  
2                 color, hex, acbench, editing, object, paint, 

In [17]:
#Save final CSV
papers_df.to_csv(
    CSV_PATH,
    index=False
)

print("CSV saved successfully.")
print(CSV_PATH)

CSV saved successfully.
/content/drive/MyDrive/research paper project/research_paper_insights.csv


In [18]:
#Verify the saved CSV

check_df = pd.read_csv(CSV_PATH)

print("CSV successfully reloaded.")
print("Rows:", len(check_df))
print("Columns:", len(check_df.columns))

check_df.head()

CSV successfully reloaded.
Rows: 5
Columns: 12


,Paper_ID,File_Name,Title,Preprint_Date,Publication_Year,Abstract,Keywords,Pages,Word_Count,Authors,Summary,TFIDF_Keywords
0,RP1,RP1.pdf,"Epidemiological Causal Graph Identification: Challenges, Identifiability and Algorithms",17 Sep 2026,NaN,"—Causal discovery from observational data is fundamental to statistics and machine learning, yet...","—Causal Discovery, Directed Acyclic Graphs, Identifiability, Exponential Family, Ordinal Distrib...",5,3938,"Sambit Mishra, Yingying Wang, Christine K. Johnson, Urbashi Mitra","Epidemiological Causal Graph Identification: Challenges, Identifiability and Algorithms Sambit M...","ordinal, family, causal, exponential, continuous, discovery, models, graphs"
1,RP2,RP2.pdf,Score Centering Stabilizes Off-Policy Reinforcement Learning,17 Sep 2026,NaN,Reinforcement learning (RL) of large language models is notoriously sensitive to small differenc...,Not provided,16,8712,"Martin Marek, Max Ryabinin",SCORE CENTERING STABILIZES OFF-POLICY REINFORCEMENT LEARNING Martin Marek & Max Ryabinin Togethe...,"training, sampling, score, quantization, importance, correction, centering, sc"
2,RP3,RP3.pdf,Paint-Anything: Unified Any-Color Control for Image Generation and Editing,17 Sep 2026,NaN,Professional design requires any-color control: the ability to specify an object’s target color ...,Not provided,29,13061,"Ji Xie, Dewei Zhou, Xinyu Huang, Zhennan Chen, Xun Wang","Paint-Anything: Unified Any-Color Control for Image Generation and Editing Ji Xie1,2, Dewei Zhou...","color, hex, acbench, editing, object, paint, generation, models"
3,RP4,RP4.pdf,Towards AI-enhanced control: a numerical technique for trajectory smoothing of a parallel robot ...,Not available,NaN,. The paper presents a numerical approach for the end-effector trajectory smoothing of a paralle...,"Trajectory Smoothing, Parallel Robot, Minimally Invasive Surgery",11,4466,"Iosif Birlescu, Alexandru Pusca, Bogdan Gherman, Calin Vaida, Ionut Zima, Damien Chablat, Doina ...",Towards AI-enhanced control: a numerical technique for trajectory smoothing of a parallel robot ...,"approach, end, command, space, effector, time, control, based"
4,RP5,RP5.pdf,Detecting Deceptive Recruitment: A Signal-theoretic Machine Learning Framework for Early Identif...,17 Sep 2026,NaN,"Deceptive online job advertisements have emerged as a primary pathway into forced labour, yet sy...","density, and visa sponsorship mentions ranking highest, followed by visual colour and texture fe...",32,10899,"Sajid Siraj, Mahnaz Hosseinzadeh, Amin Vafadarnikjoo, Shuyang Li",Detecting Deceptive Recruitment: A Signal-theoretic Machine Learning Framework for Early Identif...,"detection, risk, primary, exploiters, data, language, quality, legitimate"


In [27]:
#Final demo
demo = papers_df.iloc[0]

print("========================================")
print("RESEARCH PAPER INSIGHT GENERATOR")
print("========================================")

print("\nTITLE:")
print(demo["Title"])

print("\nAUTHORS:")
print(demo["Authors"])

print("\nPREPRINT DATE:")
print(demo["Preprint_Date"])

print("\nPAGES:")
print(demo["Pages"])

print("\nWORD COUNT:")
print(demo["Word_Count"])

print("\nABSTRACT:")
print(demo["Abstract"])

print("\nGENERATED SUMMARY:")
print(demo["Summary"])

print("\nTF-IDF KEYWORDS:")
print(demo["TFIDF_Keywords"])

print("\n========================================")
print("Output saved as research_paper_insights.csv")
print("========================================")

RESEARCH PAPER INSIGHT GENERATOR

TITLE:
Epidemiological Causal Graph Identification: Challenges, Identifiability and Algorithms

AUTHORS:
Sambit Mishra, Yingying Wang, Christine K. Johnson, Urbashi Mitra

PREPRINT DATE:
17 Sep 2026

PAGES:
5

WORD COUNT:
3938

ABSTRACT:
—Causal discovery from observational data is fundamental to statistics and machine learning, yet determining causal direction without interventions necessitates structural assumptions. Existing identifiability research primarily focuses on continuous variables under additive noise models, often neglecting mixed datasets containing ordinal scales, counts, and continuous measurements. This paper investigates causal discovery in Directed Acyclic Graphs (DAGs) where nodes follow either an ordinal distribution (via an ordered logit model) or a regular one-parameter exponential family distribution. We prove that the edge direction between an ordinal and an exponential family node is distributionally identifiable for generic 

In [36]:
#RP6 unseen-paper validation
RP6_PATH = "/content/RP6.pdf"

if os.path.exists(RP6_PATH):

    print("RP6 found. Running unseen-paper test...\n")

    rp6_text, rp6_pages = extract_pdf_text(RP6_PATH)

    rp6_abstract = extract_abstract(rp6_text)
    rp6_keywords = extract_keywords(rp6_text)
    rp6_summary = generate_summary(rp6_abstract)
    rp6_tfidf = extract_tfidf_keywords(rp6_abstract)
    rp6_date = extract_preprint_date(rp6_text)
    rp6_title, rp6_authors = extract_title_authors(RP6_PATH)

    print("===== RP6 UNSEEN PAPER TEST =====")

    print("\nTitle:")
    print(rp6_title)
    print("\nAuthors:")
    print(rp6_authors)

    print("\nPages:", rp6_pages)
    print("Preprint Date:", rp6_date)

    print("\nAbstract:")
    print(rp6_abstract)

    print("\nKeywords:")
    print(rp6_keywords)

    print("\nGenerated Summary:")
    print(rp6_summary)

    print("\nTF-IDF Keywords:")
    print(rp6_tfidf)

else:
    print("RP6.pdf not found.")
    print("Upload RP6.pdf to Colab if you want to run the unseen-paper test.")

RP6 found. Running unseen-paper test...

===== RP6 UNSEEN PAPER TEST =====

Title:
Small Transformers Track Bayesian Evidence for Latent Common Causes via a Context-Invariant Mechanism

Authors:
Amir Mohammadpour, Michael Franke

Pages: 21
Preprint Date: 28 Sep 2026

Abstract:
We present an in-depth investigation of how a form of Bayesian reasoning about common causes can emerge as a cross-contextual generalization in small, tractable transformers. Incrementing on recent work, our set-up (i) disentangles causal mech-anisms in the model from the causal structure of the true data-generating process, (ii) orients more towards natural language prediction by considering inference of latent common causes, and (iii) considers whether and how Bayesian evidence accumulation for latent common causes can be implemented in representations and mechanisms that allow for cross-context generalization to novel test cases. 1 INTRODUCTION Given the performance of large transformers, it is likely that the

In [21]:
print("""
RESEARCH PAPER INSIGHT GENERATOR
================================

Input:
Research paper PDFs

Processing:
1. PDF text extraction
2. Text cleaning
3. Abstract extraction
4. Keyword extraction
5. Extractive summarisation
6. TF-IDF keyword extraction
7. Metadata processing

Output:
Structured CSV dataset

Development papers:
RP1 - RP5

Unseen validation paper:
RP6

Next project stage:
Containerisation using Docker and Docker Compose
""")


RESEARCH PAPER INSIGHT GENERATOR

Input:
Research paper PDFs

Processing:
1. PDF text extraction
2. Text cleaning
3. Abstract extraction
4. Keyword extraction
5. Extractive summarisation
6. TF-IDF keyword extraction
7. Metadata processing

Output:
Structured CSV dataset

Development papers:
RP1 - RP5

Unseen validation paper:
RP6

Next project stage:
Containerisation using Docker and Docker Compose



In [30]:
#test cell after claude given code
import os
# from title_author_extractor import extract_title_authors   # skip this line if you pasted the functions into a cell above

# RP1-RP5 live in your Drive folder; RP6 was uploaded to /content
paths = {f"RP{i}": os.path.join(PDF_FOLDER, f"RP{i}.pdf") for i in range(1, 6)}
paths["RP6"] = "/content/RP6.pdf"

# Ground truth, checked by hand against page 1 of each PDF
truth = {
    "RP1": ("Epidemiological Causal Graph Identification: Challenges, Identifiability and Algorithms",
            ["Sambit Mishra", "Yingying Wang", "Christine K. Johnson", "Urbashi Mitra"]),
    "RP2": ("Score Centering Stabilizes Off-Policy Reinforcement Learning",
            ["Martin Marek", "Max Ryabinin"]),
    "RP3": ("Paint-Anything: Unified Any-Color Control for Image Generation and Editing",
            ["Ji Xie", "Dewei Zhou", "Xinyu Huang", "Zhennan Chen", "Xun Wang"]),
    "RP4": ("Towards AI-enhanced control: a numerical technique for trajectory smoothing of a parallel robot for pancreatic surgery",
            ["Iosif Birlescu", "Alexandru Pusca", "Bogdan Gherman", "Calin Vaida", "Ionut Zima", "Damien Chablat", "Doina Pisla"]),
    "RP5": ("Detecting Deceptive Recruitment: A Signal-theoretic Machine Learning Framework for Early Identification of Labour Exploitation",
            ["Sajid Siraj", "Mahnaz Hosseinzadeh", "Amin Vafadarnikjoo", "Shuyang Li"]),
    "RP6": ("Small Transformers Track Bayesian Evidence for Latent Common Causes via a Context-Invariant Mechanism",
            ["Amir Mohammadpour", "Michael Franke"]),
}

def norm(s):
    return " ".join(s.lower().split())

results = []
for pid, path in paths.items():
    if not os.path.exists(path):
        print(f"{pid}: file not found -> {path}")
        continue
    title, authors = extract_title_authors(path)
    exp_title, exp_authors = truth[pid]
    got_authors = [a.strip() for a in authors.split(",")]

    title_ok = norm(title) == norm(exp_title)
    missing = [a for a in exp_authors if a not in got_authors]
    extra = [a for a in got_authors if a not in exp_authors]
    authors_ok = not missing and not extra

    results.append((pid, title_ok, authors_ok))
    print(f"{pid}  title: {'PASS' if title_ok else 'FAIL'}   authors: {'PASS' if authors_ok else 'FAIL'}")
    if not title_ok:
        print(f"   expected: {exp_title}\n   got:      {title}")
    if not authors_ok:
        print(f"   missing: {missing}\n   extra:   {extra}")

n = len(results)
print(f"\nTitle accuracy:  {sum(r[1] for r in results)}/{n}")
print(f"Author accuracy: {sum(r[2] for r in results)}/{n}")

RP1  title: PASS   authors: PASS
RP2  title: PASS   authors: PASS
RP3  title: PASS   authors: PASS
RP4  title: PASS   authors: PASS
RP5  title: PASS   authors: PASS
RP6  title: PASS   authors: PASS

Title accuracy:  6/6
Author accuracy: 6/6


In [31]:
for pid, path in paths.items():
    if not os.path.exists(path):
        print(f"{pid}: file not found -> {path}\n")
        continue

    title, authors = extract_title_authors(path)

    print("=" * 60)
    print(pid)
    print("=" * 60)
    print("TITLE:")
    print(title)
    print("\nAUTHORS:")
    print(authors)
    print()

RP1
TITLE:
Epidemiological Causal Graph Identification: Challenges, Identifiability and Algorithms

AUTHORS:
Sambit Mishra, Yingying Wang, Christine K. Johnson, Urbashi Mitra

RP2
TITLE:
Score Centering Stabilizes Off-Policy Reinforcement Learning

AUTHORS:
Martin Marek, Max Ryabinin

RP3
TITLE:
Paint-Anything: Unified Any-Color Control for Image Generation and Editing

AUTHORS:
Ji Xie, Dewei Zhou, Xinyu Huang, Zhennan Chen, Xun Wang

RP4
TITLE:
Towards AI-enhanced control: a numerical technique for trajectory smoothing of a parallel robot for pancreatic surgery

AUTHORS:
Iosif Birlescu, Alexandru Pusca, Bogdan Gherman, Calin Vaida, Ionut Zima, Damien Chablat, Doina Pisla

RP5
TITLE:
Detecting Deceptive Recruitment: A Signal-theoretic Machine Learning Framework for Early Identification of Labour Exploitation

AUTHORS:
Sajid Siraj, Mahnaz Hosseinzadeh, Amin Vafadarnikjoo, Shuyang Li

RP6
TITLE:
Small Transformers Track Bayesian Evidence for Latent Common Causes via a Context-Invariant M